# 走行

4つの走行モデルをと環境モデルを用いて走行を切り替えます。

cam0: 走行用カメラ
cam1: 環境認識用カメラ

走行モデルは、右回り、左回り走行
環境モデルは、建物を認識

ミニカーバトル  2024年で使用

## Jetsonの認識

In [ ]:
import os

# ---------- 1. Jetson.GPIO 読み取り ----------
try:
    import Jetson.GPIO as GPIO
    BOARD_NAME = GPIO.gpio_pin_data.get_data()[0]
except Exception as e:
    # 失敗したら Orin Nano と決め打ち
    print(f"[WARN] Jetson モデル判定エラー: {e} → 強制的に JETSON_ORIN_NANO として続行")
    os.environ["JETSON_MODEL_NAME"] = "JETSON_ORIN_NANO"
    import Jetson.GPIO as GPIO          # もう一度ロード
    BOARD_NAME = "JETSON_ORIN_NANO"     # 確定

# ---------- 2. ボード別定義 ----------
mode_descriptions = {
    "JETSON_NX":       ["15W_2CORE", "15W_4CORE", "15W_6CORE", "10W_2CORE", "10W_4CORE"],
    "JETSON_XAVIER":   ["MAXN", "MODE_10W", "MODE_15W", "MODE_30W"],
    "JETSON_NANO":     ["MAXN", "5W"],
    "JETSON_ORIN":     ["MAXN", "MODE_15W", "MODE_30W", "MODE_40W"],
    "JETSON_ORIN_NANO":["MODE_15W", "MODE_25W", "MODE_MAX"]
}

product_names = {
    "JETSON_NX":        "Jetson Xavier NX",
    "JETSON_XAVIER":    "Jetson AGX Xavier",
    "JETSON_NANO":      "Jetson Nano",
    "JETSON_ORIN":      "Jetson AGX Orin",
    "JETSON_ORIN_NANO": "Jetson Orin Nano"
}

# (I2C バス番号, 初期 Power モードインデックス)
board_settings = {
    "JETSON_NX":        (8, 3),
    "JETSON_XAVIER":    (8, 2),
    "JETSON_NANO":      (1, 0),
    "JETSON_ORIN":      (7, 0),
    "JETSON_ORIN_NANO": (7, 2)
}

# ---------- 3. パラメータ取得 ----------
i2c_busnum, power_mode = board_settings.get(BOARD_NAME, (None, None))
mode_list       = mode_descriptions.get(BOARD_NAME, [])
product_name    = product_names.get(BOARD_NAME, "未知のボード")

# ---------- 4. 出力 ----------
if i2c_busnum is not None and 0 <= power_mode < len(mode_list):
    mode_str = mode_list[power_mode]
    print("------------------------------------------------------------")
    print(f"{product_name} を認識: I2C バス番号 = {i2c_busnum}, "
          f"Power モード = {mode_str} ({power_mode})")
    print("------------------------------------------------------------")
else:
    raise RuntimeError(f"未対応の Jetson モデル、または Power モード定義不足: {BOARD_NAME}")

In [ ]:
!echo "jetson" | sudo -S nvpmodel -m $power_mode

In [ ]:
!echo "jetson" | sudo -S nvpmodel -q

In [ ]:
!echo "jetson" | sudo -S jetson_clocks

## ログの表示用 Widget

In [ ]:
import ipywidgets
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label
import os
import glob
from IPython.display import clear_output
import traceback

l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

process_no = 0
DEBUG = False
def write_log(msg):
    global process_widget, process_no
    process_no = process_no + 1
    log_message = f"{process_no}: {msg}\n"
    process_widget.value = log_message + process_widget.value
    
    # ログファイルに書き込む
    if DEBUG:
        with open("/home/jetson/data/notebooks/logfile.log", "a") as log_file:
            log_file.write(log_message)
        
    # UIのクリアと更新
    clear_output(wait=True)

## PWMの値の読み込み

In [ ]:
import Fabo_PCA9685
import time
import pkg_resources
import smbus
import time
import json

SMBUS='smbus'
BUSNUM=i2c_busnum
SERVO_HZ=60
INITIAL_VALUE=300
bus = smbus.SMBus(BUSNUM)
PCA9685 = Fabo_PCA9685.PCA9685(bus,INITIAL_VALUE,address=0x40)
PCA9685.set_hz(SERVO_HZ)

STEERING_CH = 0
THROTTLE_CH = 1
direction = 0
REVERSE = 0
NORMAL = 1

pwm_front = 0
pwm_back = 0

with open('pwm_params.json') as f:
    json_str = json.load(f)
    
    pwm_stop = json_str["pwm_speed"]["stop"]
    pwm_front = json_str["pwm_speed"]["front"]
    pwm_back = json_str["pwm_speed"]["back"]
    pwm_left = json_str["pwm_steering"]["left"]
    pwm_center = json_str["pwm_steering"]["center"]
    pwm_right = json_str["pwm_steering"]["right"]

    
if pwm_front >= pwm_back:
    direction = REVERSE
else:
    direction = NORMAL
    
PCA9685.set_channel_value(STEERING_CH, pwm_center)
PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)

## カメラの読込

この部分でエラーが発生する場合は、Jetsonの再起動をお願いします。<br>
それでも、カメラが認識できない場合は、ケーブルの接続確認をしてください。


In [ ]:
CAM0_FPS=30
CAM1_FPS=30

In [ ]:
from csi_camera import CSICamera
import os
os.environ['OPENCV_LOG_LEVEL'] = 'SILENT'

cam0 = None
cam1 = None

def open_camera():
    global cam0, cam1
    if cam0 is not None and cam1 is not None:
        return True
    try:
        cam0 = CSICamera(capture_device=0,width=224, height=224, capture_fps=CAM0_FPS)
        cam1 = CSICamera(capture_device=1,width=224, height=224, capture_fps=CAM1_FPS)
        write_log("カメラ0,1を起動しました。")
        return True
    except Exception as e:
        # スタックトレースを含むエラーメッセージを取得
        error_message = f"Error open_camera:{e}\n{''.join(traceback.format_exception(None, e, e.__traceback__))}"
        write_log(error_message)
        close_cameras()
        return False

def close_cameras():
    global cam0, cam1
    released = False
    for cam in (cam0, cam1):
        if cam is not None:
            try:
                cam.cap.release()
                released = True
            except Exception:
                pass
    cam0 = None
    cam1 = None
    if released:
        write_log("カメラ0,1を解放しました。")

## 建物

建物のカテゴリ定義と、1つ前の建物を取得

In [ ]:
CATEGORIES = ["start","road", "right", "left", "parking", "etc"]

In [ ]:
POS_START = 1
POS_ROAD = 2
POS_RIGHT = 3
POS_LEFT = 4
POS_PARKING = 5
POS_ETC = 6

## 走行処理

In [ ]:
import threading
import torch
from utils import preprocess
import subprocess
import cv2
import time
from torch2trt import TRTModule
import subprocess
import datetime
import torch.nn.functional as F

record = False
running = False
load = False
speed_ai_flag = False
running_cam0 = False
running_cam1 = False

def map_rc(x, in_min, in_max, out_min, out_max):
    return (x - in_min) * (out_max - out_min) // (in_max - in_min) + out_min

def handle(x):
    global pwm_right,pwm_left,STEERING_CH,PCA9685
    x = map_rc(x, 224, 0, pwm_right, pwm_left)
    PCA9685.set_channel_value(STEERING_CH, x)
    
def throttle(speed, front_value, stop_value):
    global pwm_front,pwm_back,THROTTLE_CH,PCA9685
    speed = map_rc(speed, 224, 0, front_value, stop_value)
    PCA9685.set_channel_value(THROTTLE_CH, speed)

IMG_WIDTH=224

In [ ]:
def get_model(target):
    if target == "right": 
        name = "model_right"
        return name,model_run_right_trt
    elif target == "left":
        name = "model_left"
        return name,model_run_left_trt
    else:
        name = "model_left"
        return name,model_b_trt

In [ ]:
status = 0

def live_cam0():
    """
    カメラ0(フロントカメラ)からの映像を使用して走行制御を行う関数。
    大回り直進、大回り右折、小回り直進、小回り右折の4つのモデルから選べらたモデルを用いて
    ステアリング操作（handle関数呼び出し）とスロットル制御（throttle関数呼び出し）を行います。
    
    カメラ映像の記録を行う機能も備えています。
    """
    global position,cam0,target,speed_ai_flag,running_cam0,cam0,IMG_WIDTH,record,selected_model, model_run_right_trt, model_run_left_trt, preprocess,pwm_stop,save_dir0,save_dir1,num,count_cam0,fps_type,FPS_30, status
    
    try:
        count_cam0 = 1
        num = 1
        frame_count = 0
        # 処理開始時間
        start_time = time.time()
        # 走行用推論実行時間
        process_drive_time = 0
        # 走行用推論実行時間(総計)
        total_process_drive_time = 0
        
        selected_model = model_run_right_trt

        last_detect = 0
        last_lap_time = 0
        lap = 0
        
        speed = 0
        position = 1
        
        left_count = 0
        right_count = 0
        last_detect_time = 0
        model_name, model = get_model("left")
        selectec_model = model
    except Exception as e:
        write_log(f"Error live_cam0 init:{e}")

    while running_cam0:
        try:
            # カメラ画像を読込
            img0 = cam0.read()
            if record == True:
                remarked_img0 = img0.copy()
            process_drive_time = time.time()
            img0 = preprocess(img0).half()
            
            # 走行用の推論を実行
            output = selected_model(img0).detach().cpu().numpy().flatten()
            x = float(output[0]) * steering_gain_slider.value
            y = float(output[1])
            x = int(IMG_WIDTH * (x / 2.0 + 0.5))
            y = int(IMG_WIDTH * (y / 2.0 + 0.5))
            handle(x)
            
            if speed_ai_flag == False:
                speed = speed_raw_slider.value
                throttle(speed, pwm_front, pwm_stop)
            else:
                speed = float(output[3])
                speed = int(IMG_WIDTH * (speed / 2.0 + 0.5)) * speed_gain_slider.value
                throttle(speed, pwm_front, pwm_stop)
 
            # 走行用の推論実行時間を計測
            total_process_drive_time += time.time() - process_drive_time
            
            # 現在の時間を取得
            current_time = time.time()

            # 認識している状態でモデルを入れ替える(positionの値はcam1の環境情報の推論から取得）
            if position == POS_LEFT:
                model_name, model = get_model("left")
            elif position == POS_RIGHT:
                model_name, model = get_model("right")
            else:
                model_name, model = get_model("left")
            
            selected_model = model
            
            # 走行のカメラ画像を保存
            if record == True:
                name = "0_0_{:0=5}.jpg".format(count_cam0)
                image_path0 = os.path.join(save_dir0, name)
                cv2.imwrite(image_path0, remarked_img0)

            # カウンターを増加
            count_cam0 += 1
            # フレーム用のカウンターを増加
            frame_count += 1
            
            # 走行時の処理時間計測
            if time.time() - start_time > 3.0:
                fps = frame_count / 3.0
                speed_type = ""
                
                write_log(f"Cam0(走行用) FPS: {fps:.1f}, Lap: {lap}, Speed: {speed:.1f} {speed_type}, Steering Gain: {steering_gain_slider.value},  走行推論: {total_process_drive_time/(fps*3)*1000:.1f}ms ")
                frame_count = 0
                start_time = time.time() 
                total_process_drive_time = 0
            
        except Exception as e:
            # スタックトレースを含むエラーメッセージを取得
            error_message = f"Error live_cam0:{e}\n{''.join(traceback.format_exception(None, e, e.__traceback__))}"
            write_log(error_message)
            
    if record == True:
        write_log(f"画像を{count_cam0}枚の走行データを保存しました。")

## カメラ1処理

In [ ]:
last_detect_time = 0  # 最後に検出があった時刻（秒）

def check_non_detection_period(elapsed_time_ms):
    """
    特定の期間内にイベントが検出されていないかどうかをチェックする。
    elapsed_time_msはミリ秒単位で指定する。

    Args:
        elapsed_time_ms (int): 検出がないと判断する期間（ミリ秒）

    Returns:
        bool: 指定された非検出期間を超えていればTrue、そうでなければFalse
    """
    global last_detect_time
    current_time = time.time()  # 現在時刻を取得（秒）
    
    if last_detect_time == 0:  # 初期状態の場合
        last_detect_time = current_time  # 最初の検出時刻を設定

    period_time_ms = (current_time - last_detect_time) * 1000  # 経過時間をミリ秒に変換
    
    if period_time_ms > elapsed_time_ms:
        return True  # 指定された非検出期間を超えている
    else:
        return False  # 指定された非検出期間を超えていない

def update_last_detect_time():
    """
    イベント検出時に最後の検出時刻を現在時刻に更新する。
    """
    global last_detect_time
    last_detect_time = time.time()  # 現在時刻を更新（秒）

In [ ]:
def check_stop_time(current_time, target_detected_time, stop_time_ms):
    # target_detected_time および current_time は秒単位であるため、
    # ミリ秒単位での停止時間を判断するには、秒単位の差をミリ秒単位に変換する
    elapsed_time_ms = (current_time - target_detected_time) * 1000
    if elapsed_time_ms >= stop_time_ms:
        return True
    else:
        return False

In [ ]:
def live_cam1():
    """
    カメラ1からの映像を使用して環境認識を行う関数。
    クラス分類して認識した結果から現在の状態を変えていく
    """
    global position,last_detect_time,target, cam1, speed_ai_flag,running_cam1,cam1,IMG_WIDTH,record,model_env_trt,preprocess,pwm_stop,save_dir0,save_dir1,num,count_cam1,fps_type,FPS_30,status
    
    try:    
        # 認識回数
        detect_count = 0    
        # 環境情報の認識時間計測
        total_process_detect_time = 0
        # カウンター
        count_cam1 = 1
        # フレーム用カウンター
        frame_count = 0
        start_time = time.time()
        last_detect = 0
        
        target_detected_time = None
                
    except Exception as e:
        write_log(f"Error live_cam1 init:{e}")

    write_log("running_cam1")
    while running_cam1:
        try:
            # サイドカメラ画像の読込
            img1 = cam1.read()
            if record == True:
                remarked_img1 = img1.copy()
            # 環境情報の認識開始時間
            process_detect_time = time.time()
            # 画像認識は model_class_trtを使用します。
            img1 = preprocess(img1).half()
            output = model_env_trt(img1).detach()
            output = F.softmax(output, dim=1).cpu().numpy().flatten()
            category_index = output.argmax()
            # 環境情報の認識時間
            total_process_detect_time += time.time() - process_detect_time 
            
            # 現在の時間を取得
            current_time = time.time()
            
            # CATEGORIES = ["start","road", "right", "left", "parking", "etc"]
            if CATEGORIES[category_index] == "start":
                position = POS_START
            elif CATEGORIES[category_index] == "road":
                position = POS_ROAD
            elif CATEGORIES[category_index] == "right":
                position = POS_RIGHT
            elif CATEGORIES[category_index] == "left":
                position = POS_LEFT
            elif CATEGORIES[category_index] == "parking":
                position = POS_PARKING
            elif CATEGORIES[category_index] == "etc":
                position = POS_ETC        
            
            # 走行のカメラ画像を保存
            if record == True:
                name = "0_0_{:0=5}.jpg".format(count_cam1)
                image_path1 = os.path.join(save_dir1, name)
                cv2.imwrite(image_path1, remarked_img1)
                
            count_cam1 += 1
            frame_count += 1

            # 走行時の処理時間計測
            if time.time() - start_time > 3.0:
                fps = frame_count / 3.0
                speed_type = ""
                if speed_ai_flag == False:
                    speed_type = f"(固定)"
                else:
                    speed_type = f"(推論)"

                write_log(f"Cam1(環境用) target {target}, detect_count{detect_count}, prebuiling {prebuilding}, target {target}, status {status}, mode {mode}, car_id {car_id}, FPS: {fps:.1f}, 環境推論: {total_process_detect_time/(fps*3)*1000:.1f}ms")
                frame_count = 0
                start_time = time.time() 
                total_process_detect_time = 0
        except Exception as e:

            error_message = f"Error live_cam1:{e}\n{''.join(traceback.format_exception(None, e, e.__traceback__))}"
            
    if record == True:
        write_log(f"画像を{count_cam1}枚の走行データを保存しました。")

In [ ]:
def start_cameras():
    global cam0, cam1, running_cam0, running_cam1, execute_thread_cam0, execute_thread_cam1
    
    if not open_camera():
        write_log("【Error】カメラが起動できないため走行を開始しません。")
        return False
    
    # Cam0を起動
    running_cam0 = True
    execute_thread_cam0 = threading.Thread(target=live_cam0)
    execute_thread_cam0.start()
    write_log("Start cam0")

    # Cam1を起動
    running_cam1 = True
    execute_thread_cam1 = threading.Thread(target=live_cam1)
    execute_thread_cam1.start()
    write_log("Start cam1")
    return True

def setup_save_directory(base_path):
    # 指定された基本パスに基づいて保存ディレクトリを作成し、ログに記録します。
    if not os.path.exists(base_path):
        subprocess.call(['mkdir', '-p', base_path])
    write_log(f"{base_path}にデータを保存します。")

def run(change):
    global mode, running, start_time, save_dir0, save_dir1, load_model_run_left_trt, load_model_run_right_trt, load_model_env_trt
    write_log("run")
    if not (load_model_run_right_trt and load_model_run_left_trt and load_model_env_trt):
        write_log("モデルが読み込まれていません")
        return
    if not running:
        write_log("AIが起動しました。")
        if record:
            if name_widget.value != "":
                base_path = "camera/" + name_widget.value
                # Cam0とCam1の保存先を設定
                save_dir0 = f"{base_path}_cam0/xy/"
                save_dir1 = f"{base_path}_cam1/xy/"
                
                setup_save_directory(save_dir0)
                setup_save_directory(save_dir1)
            else:
                write_log("【Error】 映像の保存先を入力してください。")
                return
        write_log("カメラを起動中...")
        if not start_cameras():
            return
        start_time = time.time()
        
        
def stop(change):
    global running_cam0,running_cam1,execute_thread_cam0,execute_thread_cam1,end_time,start_time,count_cam0,pwm_stop,mode_running
    if running_cam0 == True:
        try:
            end_time = time.time() - start_time
            fps = count_cam0/int(end_time)
            process_time = int((end_time/count_cam0)*1000)
        except:
            fps = -1
            process_time = -1
        mode_running = False
        write_log("AIを停止しました。")
        write_log("処理結果:FPS: " + str(round(fps,2)) + ",処理回数: " + str(count_cam0) + ",　処理時間(1回平均値): " + str(process_time) + " ms")
        running = False
        running_cam0 = False
        running_cam1 = False
        PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)
        try:    
            execute_thread_cam0.join()
            execute_thread_cam1.join()
        except:
            write_log("Thread joinでエラー(すでにthreadが存在しない")
        PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)
        close_cameras()
        
    else:
        PCA9685.set_channel_value(THROTTLE_CH, pwm_stop)
        write_log("現在AIは動いていません。")
        close_cameras()

## UI

In [ ]:
model_run_right_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_run_right_time_widget = ipywidgets.Label(description='作成日時')
load_run_right_button = ipywidgets.Button(description='走行モデルを読込み')
model_run_left_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_run_left_time_widget = ipywidgets.Label(description='作成日時')
load_run_left_button = ipywidgets.Button(description='走行モデルを読込み')

model_env_widget = ipywidgets.Dropdown(options=[],description='モデル')
model_env_time_widget = ipywidgets.Label(description='作成日時')

load_env_button = ipywidgets.Button(description='環境モデルを読込み')

run_button = ipywidgets.Button(description='走行開始')
stop_button = ipywidgets.Button(description='停止')

name_widget = ipywidgets.Text(description='映像の保存名')
record_box = ipywidgets.Checkbox(False, description='録画')
speed_gain_slider = ipywidgets.FloatSlider(description='Speed gain', min=0.1, max=3.0, step=0.05, value=0.45, orientation='horizontal')
speed_low_slider = ipywidgets.FloatSlider(description='Speed low', min=0, max=100, step=1, value=20, orientation='horizontal')
speed_raw_slider = ipywidgets.IntSlider(description='Speed raw', min=1, max=224, step=1, value=80, orientation='horizontal')
steering_gain_slider = ipywidgets.FloatSlider(description='Steering gain', min=0.1, max=3.0, step=0.1, value=2.5, orientation='horizontal')
speed_dropbox = ipywidgets.Dropdown(options=["推論値","固定値"], description='Speed')

building_dropbox = ipywidgets.Dropdown(options=CATEGORIES, description='Building')

In [ ]:
import numpy as np

def load_model(widget, model_var_name):
    try:
        write_log(f"{widget.value}の読込を実行します(初回は時間がかかります)。")
        model = TRTModule()
        model.load_state_dict(torch.load(widget.value,weights_only=True))
        model(preprocess(np.zeros((224, 224, 3)).astype(np.uint8)))
        write_log(f"{widget.value}の読込に成功しました。")
        globals()[model_var_name] = model  # 成功した場合、グローバル変数にモデルをセット
        load_flag_var_name = f"load_{model_var_name}"
        write_log(f"{load_flag_var_name}の読込に成功しました。")
        globals()[load_flag_var_name] = True  # 対応するフラグをTrueにセット
        get_jetson_nano_memory_usage()
    except Exception as e:
        write_log(f"【Error】 {e} : {widget.value} の読込に失敗しました。")
        
# モデル読み込み関数を各ボタンのクリックイベントにバインドする例
load_run_right_button.on_click(lambda change: load_model(model_run_right_widget, 'model_run_right_trt'))
load_run_left_button.on_click(lambda change: load_model(model_run_left_widget, 'model_run_left_trt'))

load_env_button.on_click(lambda change: load_model(model_env_widget, 'model_env_trt'))

In [ ]:
def model_list(type):
    try:
        files = glob.glob(type + '/*.pth', recursive=True)
         
        # 走行用の回帰モデル
        if type == "model_trt":
            model_run_right_widget.options = files
            model_run_left_widget.options = files
            
        # クラス分類モデル
        elif type == "model_class_trt":
            model_env_widget.options = files
        
    except Exception as e:
        if type == "model_trt":
            model_run_right_widget.options = []
            model_run_left_widget.options = []
        elif type == "model_class_trt":
            model_env_widget.options = []
        write_log(f"Error model_list:{e}")
        
model_list("model_trt")
model_list("model_class_trt")

def update_model_time_widget(widget, time_widget):
    """指定されたウィジェットのファイル選択が変更された際の処理。ファイルの作成時間を表示ウィジェットに設定する。"""
    file = widget.value
    try:
        ts = os.path.getctime(file)
        d = datetime.datetime.fromtimestamp(ts)
        s = d.strftime('%Y-%m-%d %H:%M:%S')
        time_widget.value = s
    except Exception as e:
        time_widget.value = "Error update_model_time_widget: " + str(e)

# 各モデル選択ウィジェットの変更を監視し、対応する時刻表示ウィジェットを更新
model_run_right_widget.observe(lambda change: update_model_time_widget(model_run_right_widget, model_run_right_time_widget), names='value')
model_run_left_widget.observe(lambda change: update_model_time_widget(model_run_left_widget, model_run_left_time_widget), names='value')

model_env_widget.observe(lambda change: update_model_time_widget(model_env_widget, model_env_time_widget), names='value')

def on_video(change):
    global record
    record^=True


def on_fixed_value_change(change):
    global speed_ai_flag, speed_gain_slider, speed_raw_slider
    if change['new'] == "固定値":
        speed_gain_slider.layout.visibility = 'hidden'
        speed_low_slider.layout.visibility = 'hidden'
        speed_raw_slider.layout.visibility = 'visible'
        speed_gain_slider.disabled = True
        speed_raw_slider.disabled = False
        speed_ai_flag = False
    elif change['new'] == "推論値":
        speed_gain_slider.layout.visibility = 'visible'
        speed_low_slider.layout.visibility = 'visible'
        speed_raw_slider.layout.visibility = 'hidden'
        speed_gain_slider.disabled = False
        speed_raw_slider.disabled = True
        speed_ai_flag = True

speed_gain_slider.layout.visibility = 'visible'
speed_low_slider.layout.visibility = 'visible'
speed_raw_slider.layout.visibility = 'hidden'
speed_gain_slider.disabled = False
speed_raw_slider.disabled = True
speed_ai_flag = True
speed_dropbox.observe(on_fixed_value_change, names='value')

run_button.on_click(run)
stop_button.on_click(stop)
record_box.observe(on_video)

In [ ]:
import subprocess
import re

used_memory_widget = ipywidgets.IntText(description='Useメモリ', value=1)
total_memory_widget = ipywidgets.IntText(description='全メモリ', value=1)
memory_button = ipywidgets.Button(description='使用メモリ量の取得')

def get_jetson_nano_memory_usage(event=None):
    command = 'tegrastats'
    try:
        process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        
        mem_usage_pattern = re.compile(r'RAM (\d+)/(\d+)MB')
        
        max_lines_to_read = 10
        for _ in range(max_lines_to_read):
            line = process.stdout.readline()
            if not line:
                break 
            matches = mem_usage_pattern.search(line)
            if matches:
                used_memory_widget.value = int(matches.group(1))
                total_memory_widget.value = int(matches.group(2))
                process.kill()
                return
        
        process.kill()  
        return

    except subprocess.CalledProcessError as e:
        return

get_jetson_nano_memory_usage()
memory_button.on_click(get_jetson_nano_memory_usage)

走行までの流れは以下の通りです。

1. <b>走行モデルを指定してLoadする</b><br>
2. <b>環境モデルを指定してLoadする</b><br>
3. <b>Speedに固定値, Speedに推論値のいずれかを選択する</b><br>
固定値を選んだ場合は固定の速度で走り続けます。推論値を選んだ推論結果を速度に反映します。速度用のアノテーションは13_annotation.ipynbで追加可能です。<br>
4. <b>[オプション] 走行動画を録画する場合は、録画にチェックマークをいれて、保存ファイル名を指定する</b><br>
./runフォルダに保存<br>
5. <b>走行開始ボタンを押して、プロポの裏側のボタンを押して AIモードで自動走行開始する</b><br>
6. <b>終了時は、停止ボタンを押す</b><br>
録画のチェックマークがついている場合は、停止で録画も終了<br>
Speed Inferenceのチェックマークがついている場合は、スロットル量の推論も有効になる<br>
<br>
カメラが60fpsで動いている場合は16ms以内、カメラが30fpsで動いている場合は、33ms以内での処理完了が正常な挙動となります。

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')
title1 = ipywidgets.HTML('<b>【1.使用する走行モデルA(Right)】</b> TensoRTに変換済みのモデルをLoadします。')
title2 = ipywidgets.HTML('<b>【2.使用する走行モデルB(Left)】</b> TensoRTに変換済みのモデルをLoadします。')
title3 = ipywidgets.HTML('<b>【3.使用する環境モデルClass】</b> TensoRTに変換済みのモデルをLoadします。')
title4 = ipywidgets.HTML('<b>【4.使用する走行モデルC(Parking)】</b> TensoRTに変換済みのモデルをLoadします。')

title5 = ipywidgets.HTML('<b>【5.Steeringゲイン】</b> Steeringのゲイン調整します。周りが悪い時は値を1.0以上にします。')
title6 = ipywidgets.HTML('<b>【6.速度】</b> 速度は固定値か、推論から反映かが選べます。')
title7 = ipywidgets.HTML('<b>【7.録画】</b> 走行中の映像を録画したい場合はチェックマークを選択し、保存データセット名を指定してください。')
title8 = ipywidgets.HTML('<b>【8.走行の開始】</b> 走行開始を押す前にタイヤが空転していのを確認してください。プロポの裏のボタンを変換しAIモードにすると動き始めます。')

data_collection_widget = ipywidgets.VBox([
    separator,
    title1,
    ipywidgets.HBox([model_run_right_widget,model_run_right_time_widget,load_run_right_button]),
    process_widget,
    separator,
    title2,
    ipywidgets.HBox([model_run_left_widget,model_run_left_time_widget,load_run_left_button]),
    process_widget,
    separator,
    title3,
    ipywidgets.HBox([model_env_widget,model_env_time_widget,load_env_button]),
    process_widget,
    separator,
    title5,
    ipywidgets.HBox([steering_gain_slider]),
    separator,
    title6,
    ipywidgets.HBox([speed_dropbox]),
    ipywidgets.HBox([speed_raw_slider,speed_low_slider, speed_gain_slider]),
    separator,
    title7,
    ipywidgets.HBox([record_box,name_widget]),
    separator,
    title8,
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    ipywidgets.HBox([run_button, stop_button]),
    process_widget,
    separator,
])
display(data_collection_widget)